Import libraries

In [1]:
#import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from cfrnet import CFRnet
import sys
from pathlib import Path
project_root = Path("/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from utils import seed_everything
from metrics import auuc, auqc, lift, krcc

In [2]:
%time train_df = pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Men/train_men.csv")
%time test_df =  pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Men/test_men.csv")
%time val_df = pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Men/val_men.csv")

CPU times: user 22.1 ms, sys: 9.9 ms, total: 32 ms
Wall time: 34.7 ms
CPU times: user 12.9 ms, sys: 3 ms, total: 15.9 ms
Wall time: 17.5 ms
CPU times: user 3.9 ms, sys: 2.99 ms, total: 6.89 ms
Wall time: 7.8 ms


In [3]:
in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
       'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = ['conversion']
treatment_feature = ['treatment']

In [4]:
X_train = train_df[in_features].values.astype(float) # type: ignore
y_train = train_df[label_feature].values.astype(float) # type: ignore
t_train = train_df[treatment_feature].values.astype(float) # type: ignore

X_test = test_df[in_features].values.astype(float) # type: ignore
y_test = test_df[label_feature].values.astype(float) # type: ignore
t_test = test_df[treatment_feature].values.astype(float) # type: ignore

X_val = val_df[in_features].values.astype(float) # type: ignore
y_val = val_df[label_feature].values.astype(float) # type: ignore
t_val = val_df[treatment_feature].values.astype(float) # type: ignore

In [5]:
print('X_train[:10]', X_train[:1].astype(float))

X_train[:10] [[-0.21435131  1.6331766   1.0667411   0.90252386 -1.1010233   1.07039981
   1.00043033  2.70003843 -0.88552759 -0.88616046]]


In [6]:
print('y_train[:10]', y_train[:1].astype(float))

y_train[:10] [[0.]]


In [7]:
# Transform to tensor
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

x_men_train_t = to_tensor(X_train)
x_men_val_t = to_tensor(X_val)
x_men_test_t = to_tensor(X_test)

y_men_train_t = to_tensor(y_train).reshape(-1, 1)
y_men_val_t = to_tensor(y_val).reshape(-1, 1)
y_men_test_t = to_tensor(y_test).reshape(-1, 1)

# t_train/t_val/t_test cũng tương tự
t_men_train_t = to_tensor(t_train.astype(float)).reshape(-1, 1)
t_men_val_t = to_tensor(t_val.astype(float)).reshape(-1, 1)
t_men_test_t = to_tensor(t_test.astype(float)).reshape(-1, 1)

# Data loader
train_dataset = TensorDataset(x_men_train_t, t_men_train_t, y_men_train_t)
val_dataset = TensorDataset(x_men_val_t, t_men_val_t, y_men_val_t)
test_dataset = TensorDataset(x_men_test_t, t_men_test_t, y_men_test_t)

batch_size = 6400
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)

print("-------------------------------------------------------------")
print("✅ Completed transform to tensor ✅")
print(f"Shape of train: x={x_men_train_t.shape}; y={y_men_train_t.shape}; t={t_men_train_t.shape}")
print(f"Shape of val: x={x_men_val_t.shape}; y={y_men_val_t.shape}; t={t_men_val_t.shape}")
print(f"Shape of test: x={x_men_test_t.shape}; y={y_men_test_t.shape}; t={t_men_test_t.shape}")

-------------------------------------------------------------
✅ Completed transform to tensor ✅
Shape of train: x=torch.Size([25567, 10]); y=torch.Size([25567, 1]); t=torch.Size([25567, 1])
Shape of val: x=torch.Size([4262, 10]); y=torch.Size([4262, 1]); t=torch.Size([4262, 1])
Shape of test: x=torch.Size([12784, 10]); y=torch.Size([12784, 1]); t=torch.Size([12784, 1])


In [8]:
epochs = 150
alpha = 1
lr = 1e-3
wd = 1e-5
method = "mmd_rbf"
early_stop_metric = "qini"
ema = True
ema_alpha = 0.15
patience = 20
early_stop_start = 0
shared_hidden = 200
outcome_hidden = 100
activation = torch.nn.ReLU
outcome_dropout = 0.0
shared_dropout = 0.0
print (f" epochs = {epochs}")
print (f" method = {method}")
print (f" alpha = {alpha}")
print (f" learning rate = {lr}")
print (f" weight decay = {wd}")
print (f" early stop = {early_stop_metric}")
print (f" use ema = {ema}")
print (f" ema alpha = {ema_alpha}")
print (f" patience = {patience}")
print (f" shared hidden = {shared_hidden}")
print (f" outcome hidden = {outcome_hidden}")
print (f"activation = {activation}")

 epochs = 150
 method = mmd_rbf
 alpha = 1
 learning rate = 0.001
 weight decay = 1e-05
 early stop = qini
 use ema = True
 ema alpha = 0.15
 patience = 20
 shared hidden = 200
 outcome hidden = 100
activation = <class 'torch.nn.modules.activation.ReLU'>


In [9]:
import pandas as pd
import numpy as np
import torch
import io
import optuna
from contextlib import redirect_stdout, redirect_stderr

# Minimize Optuna console noise
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Optuna search config (validation only)
seeds = [412312, 42, 1874, 902745, 1]
n_trials = 50
tpe_sampler_seed = 412312
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def objective(trial):
    grid_lr = trial.suggest_float("lr", 5e-5, 5e-4, log=True)
    grid_wd = trial.suggest_float("weight_decay", 1e-5, 1e-4, log=True)
    grid_alpha = trial.suggest_categorical("alpha", [0.1, 0.5, 1.0])
    grid_method = trial.suggest_categorical("method", ["mmd_rbf", "mmd_linear"])
    val_scores = []
    for SEED in seeds:
        seed_everything(SEED)

        cfr = CFRnet(
            input_dim=x_men_train_t.shape[1],
            epochs=epochs,
            learning_rate=grid_lr,
            weight_decay=grid_wd,
            alpha=grid_alpha,
            method=grid_method,
            use_ema=ema,
            ema_alpha=ema_alpha,
            patience=patience,
            shared_hidden=shared_hidden,
            outcome_hidden=outcome_hidden,
            outcome_dropout=outcome_dropout,
            shared_dropout=shared_dropout,
            early_stop_metric=early_stop_metric,
            early_stop_start_epoch=early_stop_start,
            activation=activation
        )

        # Silence CFRNet epoch logs
        with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            cfr.fit(train_loader, val_loader)

        x_val_device = x_men_val_t.to(device)
        y0_val_p, y1_val_p= cfr.predict(x_val_device)
        uplift_val = (y1_val_p - y0_val_p).detach().cpu().numpy().flatten()
        y_val_true_np = y_men_val_t.detach().cpu().numpy().flatten()
        t_val_true_np = t_men_val_t.detach().cpu().numpy().flatten()

        current_val_auqc = auqc(y_val_true_np, t_val_true_np, uplift_val, bins=100, plot=False)
        val_scores.append(current_val_auqc)

    mean_val_auqc = float(np.mean(val_scores))
    std_val_auqc = float(np.std(val_scores, ddof=1)) if len(val_scores) > 1 else 0.0
    trial.set_user_attr("std_Val_AUQC", std_val_auqc)
    return mean_val_auqc

sampler = optuna.samplers.TPESampler(seed=tpe_sampler_seed)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

# 2. Persist search results in notebook variables
trial_rows = []
for t in study.trials:
    if t.value is None:
        continue
    trial_rows.append({
        "trial": t.number,
        "lr": t.params["lr"],
        "weight_decay": t.params["weight_decay"],
        "alpha": t.params["alpha"],
        "method": t.params["method"],
        "mean_Val_AUQC": float(t.value),
        "std_Val_AUQC": float(t.user_attrs.get("std_Val_AUQC", np.nan))
    })

df_grid = pd.DataFrame(trial_rows).sort_values("mean_Val_AUQC", ascending=False).reset_index(drop=True)
best_params = study.best_params
best_cfg = pd.Series({
    "lr": float(best_params["lr"]),
    "weight_decay": float(best_params["weight_decay"]),
    "alpha": float(best_params["alpha"]),
    "method": t.params["method"],
    "mean_Val_AUQC": float(study.best_value),
    "std_Val_AUQC": float(study.best_trial.user_attrs.get("std_Val_AUQC", np.nan))
})

/home/ducvu0904/miniconda3/envs/ai/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
  0%|          | 0/50 [00:00<?, ?it/s]

Locked random seed: 412312


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:275: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Locked random seed: 42


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:275: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


...

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:275: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Locked random seed: 902745


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:275: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Locked random seed: 1


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:275: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
Best trial: 36. Best value: 0.754361: 100%|██████████| 50/50 [45:40<00:00, 54.80s/it]


In [10]:
from IPython.display import display

if 'study' not in globals():
    print('Run Cell 10 (Optuna tuning) first.')
else:
    print(f"Best trial: {study.best_trial.number}")
    print(f"Best mean_Val_AUQC: {study.best_value:.6f}")
    print(f"Best params: {study.best_params}")

if 'best_cfg' in globals():
    print('\nBest config table:')
    display(best_cfg.to_frame().T)
else:
    print('\nbest_cfg not found.')

if 'df_grid' in globals():
    print('\nTop 10 trials:')
    display(df_grid.head(10))
else:
    print('\ndf_grid not found.')

if 'df_results' in globals():
    print('\nPer-seed test results:')
    display(df_results)
    print('\nTest metrics mean ± std:')
    display(df_results.drop(columns='Seed').agg(['mean', 'std']))

Best trial: 36
Best mean_Val_AUQC: 0.754361
Best params: {'lr': 0.00040243290444530894, 'weight_decay': 1.891767602044244e-05, 'alpha': 0.5, 'method': 'mmd_rbf'}

Best config table:


,lr,weight_decay,alpha,method,mean_Val_AUQC,std_Val_AUQC
0,0.000402,0.000019,0.5,mmd_rbf,0.754361,0.016893



Top 10 trials:


,trial,lr,weight_decay,alpha,method,mean_Val_AUQC,std_Val_AUQC
0,36,0.000402,0.000019,0.5,mmd_rbf,0.754361,0.016893
1,27,0.000405,0.000019,1.0,mmd_rbf,0.754239,0.017987
2,26,0.000414,0.000018,1.0,mmd_rbf,0.753079,0.016277
3,47,0.000414,0.000016,0.5,mmd_rbf,0.751068,0.016185
4,42,0.000423,0.000017,0.5,mmd_rbf,0.749082,0.015264
5,43,0.000421,0.000016,0.5,mmd_rbf,0.748954,0.015149
6,32,0.000428,0.000018,0.5,mmd_rbf,0.748598,0.013108
7,29,0.000393,0.000019,0.5,mmd_rbf,0.746671,0.016157
8,14,0.000393,0.000017,1.0,mmd_rbf,0.746522,0.016056
9,11,0.000439,0.000010,1.0,mmd_rbf,0.745007,0.017047


In [12]:
import pandas as pd
import numpy as np
import torch

# 1. Cấu hình
seeds = [412312, 42, 1874, 902745, 1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_runs = []

print(f"🔄 Đang huấn luyện TARNet trên {len(seeds)} seeds khác nhau... Vui lòng đợi.")

# 2. Vòng lặp chạy (Chỉ xử lý, không in kết quả lẻ)
for SEED in seeds:
    seed_everything(SEED)
    
    # Khởi tạo lại mô hình
    cfrnet = CFRnet(
        input_dim=x_men_train_t.shape[1], epochs=epochs, learning_rate=4.02E-04, 
        alpha=0.5, method="mmd_rbf",
        weight_decay=1.90E-05, use_ema=ema, ema_alpha=ema_alpha, patience=patience,
        shared_hidden=shared_hidden, outcome_hidden=outcome_hidden,
        early_stop_metric=early_stop_metric, early_stop_start_epoch=early_stop_start,
        activation=activation
    )
    
    cfrnet.fit(train_loader, val_loader)
    
    # Predict
    x_men_test_t_on_device = x_men_test_t.to(device)
    y0_pred, y1_pred = cfrnet.predict(x_men_test_t_on_device)
    
    uplift_pred = (y1_pred - y0_pred).cpu().numpy().flatten()
    y_true = y_men_test_t.cpu().numpy().flatten()
    t_true = t_men_test_t.cpu().numpy().flatten()
    
    # Tính toán
    ate_pred = uplift_pred.mean()
    ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()
    
    all_runs.append({
        'Seed': SEED,
        'AUUC': auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'AUQC': auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'Lift': lift(y_true, t_true, uplift_pred, h=0.3),
        'KRCC': krcc(y_true, t_true, uplift_pred, bins=100),
        'ATE_Err': abs(ate_pred - ate_true)
    })
    print(f"  ✔️ Đã xong Seed {SEED}")

# 3. HIỂN THỊ KẾT QUẢ TỔNG HỢP (Print 1 lúc)
df_results = pd.DataFrame(all_runs)

print("\n" + "="*85)
print(f"{'CHI TIẾT TỪNG SEED':^85}")
print("="*85)
# Sử dụng to_string để in bảng đẹp mắt
print(df_results.to_string(index=False, formatters={
    'AUUC': '{:,.4f}'.format, 'AUQC': '{:,.4f}'.format, 
    'Lift': '{:,.4f}'.format, 'KRCC': '{:,.4f}'.format, 'ATE_Err': '{:,.4f}'.format
}))

# 4. Tính toán Mean và Std
mean_res = df_results.drop(columns='Seed').mean()
std_res = df_results.drop(columns='Seed').std()

print("="*85)
print(f"{'KẾT QUẢ TRUNG BÌNH (MEAN ± STD)':^85}")
print("-" * 85)
summary_data = []
for metric in ['AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
    print(f"{metric:<10}: {mean_res[metric]:.4f} ± {std_res[metric]:.4f}")

print("="*85)

🔄 Đang huấn luyện TARNet trên 5 seeds khác nhau... Vui lòng đợi.
Locked random seed: 412312
🔃🔃🔃Begin training Dragonnet🔃🔃🔃
📊 Early Stop Metric: QINI
📊 Early Stop Start Epoch: 1
📊 Strategy: Two-Stage EMA Filter (alpha=0.15)
   EMA filters noise spikes, Raw Qini determines peak height
   Select checkpoint: raw_qini is highest AND raw_qini >= ema_qini
Epoch 1/150 | Loss: 0.4911 | Val Loss: 0.4858 | Val Qini: 0.5619 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5619 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 0.4700 | Val Loss: 0.4638 | Val Qini: 0.7538 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5907 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 0.4416 | Val Loss: 0.4328 | Val Qini: 0.7028 ✓ above trend but not peak (patience: 1/20)EMA Trend: 0.6075 | ✓ above trend but not peak (patience: 1/20)
Epoch 4/150 | Loss: 0.3966 | Val Loss: 0.3808 | Val Qini: 0.7156 ✓ above trend but not peak (patience: 2/20)EMA Trend: 0.6237 | ✓ above trend but not peak (patience: 2/20)
Epoch 5/150 | Loss: 0.3170 | Val 

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:275: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 0.5046 | Val Loss: 0.4980 | Val Qini: 0.4312 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.4312 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 0.4745 | Val Loss: 0.4646 | Val Qini: 0.3541 (patience: 1/20)EMA Trend: 0.4196 | (patience: 1/20)
Epoch 3/150 | Loss: 0.4262 | Val Loss: 0.4086 | Val Qini: 0.3040 (patience: 2/20)EMA Trend: 0.4023 | (patience: 2/20)
Epoch 4/150 | Loss: 0.3408 | Val Loss: 0.3132 | Val Qini: 0.2729 (patience: 3/20)EMA Trend: 0.3829 | (patience: 3/20)
Epoch 5/150 | Loss: 0.2148 | Val Loss: 0.1783 | Val Qini: 0.2727 (patience: 4/20)EMA Trend: 0.3663 | (patience: 4/20)
Epoch 6/150 | Loss: 0.0833 | Val Loss: 0.0658 | Val Qini: 0.7662 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.4263 | ⭐ NEW BEST (peak ≥ trend)
Epoch 7/150 | Loss: 0.0289 | Val Loss: 0.0248 | Val Qini: 0.7363 ✓ above trend but not peak (patience: 1/20)EMA Trend: 0.4728 | ✓ above trend but not peak (patience: 1/20)
Epoch 8/150 | Loss: 0.0173 | Val Loss: 0.0186 | Val Qini: 0.7297 ✓ above trend but n

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:275: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 0.4803 | Val Loss: 0.4725 | Val Qini: 0.3781 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3781 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 0.4464 | Val Loss: 0.4347 | Val Qini: 0.5125 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3983 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 0.3892 | Val Loss: 0.3695 | Val Qini: 0.5932 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.4275 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Loss: 0.2929 | Val Loss: 0.2622 | Val Qini: 0.7460 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.4753 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Loss: 0.1628 | Val Loss: 0.1291 | Val Qini: 0.7697 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5194 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/150 | Loss: 0.0574 | Val Loss: 0.0407 | Val Qini: 0.7634 ✓ above trend but not peak (patience: 1/20)EMA Trend: 0.5560 | ✓ above trend but not peak (patience: 1/20)
Epoch 7/150 | Loss: 0.0226 | Val Loss: 0.0199 | Val Qini: 0.7600 ✓ above trend but not peak (patience: 2/20)EMA Trend: 0.5866 | ✓ above trend but no

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:275: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 0.4905 | Val Loss: 0.4812 | Val Qini: 0.7256 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7256 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 0.4478 | Val Loss: 0.4337 | Val Qini: 0.7157 (patience: 1/20)EMA Trend: 0.7241 | (patience: 1/20)
Epoch 3/150 | Loss: 0.3800 | Val Loss: 0.3572 | Val Qini: 0.7324 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7254 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Loss: 0.2729 | Val Loss: 0.2402 | Val Qini: 0.7524 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7294 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Loss: 0.1358 | Val Loss: 0.1055 | Val Qini: 0.5757 (patience: 1/20)EMA Trend: 0.7064 | (patience: 1/20)
Epoch 6/150 | Loss: 0.0399 | Val Loss: 0.0318 | Val Qini: 0.2822 (patience: 2/20)EMA Trend: 0.6427 | (patience: 2/20)
Epoch 7/150 | Loss: 0.0216 | Val Loss: 0.0188 | Val Qini: 0.2787 (patience: 3/20)EMA Trend: 0.5881 | (patience: 3/20)
Epoch 8/150 | Loss: 0.0182 | Val Loss: 0.0181 | Val Qini: 0.2794 (patience: 4/20)EMA Trend: 0.5418 | (patience: 4/20)
Ep

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:275: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 0.4934 | Val Loss: 0.4868 | Val Qini: 0.5931 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5931 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 0.4620 | Val Loss: 0.4517 | Val Qini: 0.6722 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.6050 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 0.4122 | Val Loss: 0.3949 | Val Qini: 0.5092 (patience: 1/20)EMA Trend: 0.5906 | (patience: 1/20)
Epoch 4/150 | Loss: 0.3256 | Val Loss: 0.2962 | Val Qini: 0.1659 (patience: 2/20)EMA Trend: 0.5269 | (patience: 2/20)
Epoch 5/150 | Loss: 0.1927 | Val Loss: 0.1573 | Val Qini: 0.1453 (patience: 3/20)EMA Trend: 0.4697 | (patience: 3/20)
Epoch 6/150 | Loss: 0.0708 | Val Loss: 0.0488 | Val Qini: 0.4389 (patience: 4/20)EMA Trend: 0.4651 | (patience: 4/20)
Epoch 7/150 | Loss: 0.0243 | Val Loss: 0.0202 | Val Qini: 0.6869 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.4984 | ⭐ NEW BEST (peak ≥ trend)
Epoch 8/150 | Loss: 0.0169 | Val Loss: 0.0182 | Val Qini: 0.7213 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5318 | ⭐ NEW BEST

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:275: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
